In [2]:
import pandas as pd

filename = './merged_data/places.csv'

df = pd.read_csv(filename)

len(pd.read_csv(filename))

2557

In [17]:
df

,source_id,continent,country,city,section,category,name,address,description,url,image_url
0,paragraph_000015,North America,United States,New York,Accommodation,Hotels,Night Hotel Broadway (From $132.00),"215 West 94th Street, Upper West Side, New Yor...",On the Upper West Side of Manhattan and within...,https://www.booking.com/hotel/us/days-inn-broa...,media/image1.jpg
1,paragraph_000023,North America,United States,New York,Accommodation,Hotels,"The Mayfair Hotel Times Square, an Ascend Coll...","242 West 49th Street, New York, NY 10019, Unit...",Located in the heart of Manhattan’s Theater Di...,https://www.booking.com/hotel/us/mayfair-new-y...,media/image2.jpg
2,paragraph_000029,North America,United States,New York,Accommodation,Hotels,Conrad New York Downtown (From $561),"102 North End Avenue, Battery Park, New York, ...",With 2000 pieces of artwork on display and an ...,https://www.booking.com/hotel/us/e-suites-new-...,media/image3.jpg
3,paragraph_000040,North America,United States,New York,Accommodation,Bed&Breakfast,American Dream Guest House (From $150.00),"532 West 48th Street, Hell's Kitchen, New York...",Comfortable Accommodations: American Dream Gue...,https://www.booking.com/hotel/us/american-drea...,media/image4.jpg
4,paragraph_000048,North America,United States,New York,Accommodation,Bed&Breakfast,Interfaith 21st Street (From $247.00),"246 West 21st Street, Chelsea, New York, NY 10...",Comfortable Accommodations: Interfaith 21st St...,https://www.booking.com/hotel/us/interfaith-21...,media/image5.jpg
...,...,...,...,...,...,...,...,...,...,...,...
2552,paragraph_019408,Africa,United Arab Emirates,Sharjah,Extreme sport,Extreme sport,XQuarry Off-Road and Adventure Park (Adventure...,"Off-roading, obstacle course, mountain biking,...",NaN,https://xquarry.com/,no_image
2553,paragraph_019412,Africa,United Arab Emirates,Sharjah,Extreme sport,Extreme sport,Desert EJet (AED 250 per person),Water karting in the middle of the desert. 10 ...,NaN,NaN,no_image
2554,paragraph_019415,Africa,United Arab Emirates,Sharjah,Extreme sport,Extreme sport,Sky Adventures - Tandem Paragliding (AED 750 p...,Tandem paragliding flight with instructor over...,NaN,https://sky-adventures.ae/,no_image
2555,paragraph_019419,Africa,United Arab Emirates,Sharjah,Extreme sport,Extreme sport,Sky Adventures - Guided Flight (AED 350 per fl...,"For licensed paragliders. Weather observation,...",NaN,https://sky-adventures.ae/,no_image


In [14]:
import pandas as pd

df = pd.read_csv("merged_data/places.csv").fillna("")

continents = (
    df[["continent"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
continents["id"] = continents.index + 1
continents = continents[["id", "continent"]]
continents = continents.rename(columns={"continent": "name"})


countries = (
    df[["continent", "country"]]
    .drop_duplicates()
    .merge(continents, left_on="continent", right_on="name")
    [["id", "country"]]
    .rename(columns={"id": "continent_id", "country": "name"})
    .drop_duplicates()
    .reset_index(drop=True)
)
countries["id"] = countries.index + 1
countries = countries[["id", "continent_id", "name"]]


cities = (
    df[["country", "city"]]
    .drop_duplicates()
    .merge(countries, left_on="country", right_on="name")
    [["id", "city"]]
    .rename(columns={"id": "country_id", "city": "name"})
    .drop_duplicates()
    .reset_index(drop=True)
)
cities["id"] = cities.index + 1
cities = cities[["id", "country_id", "name"]]


sections = (
    df[["section"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
sections["id"] = sections.index + 1
sections = sections[["id", "section"]].rename(columns={"section": "name"})


categories = (
    df[["section", "category"]]
    .drop_duplicates()
    .merge(sections, left_on="section", right_on="name")
    [["id", "category"]]
    .rename(columns={"id": "section_id", "category": "name"})
    .drop_duplicates()
    .reset_index(drop=True)
)
categories["id"] = categories.index + 1
categories = categories[["id", "section_id", "name"]]

In [16]:
places = df.copy()

places = places.merge(
    continents,
    left_on="continent",
    right_on="name",
    how="left",
).rename(columns={"id": "continent_id"}).drop(columns=["name_y"])

places = places.rename(columns={"name_x": "name"})

places = places.merge(
    countries,
    left_on=["country", "continent_id"],
    right_on=["name", "continent_id"],
    how="left",
).rename(columns={"id": "country_id"}).drop(columns=["name_y"])

places = places.rename(columns={"name_x": "name"})

places = places.merge(
    cities,
    left_on=["city", "country_id"],
    right_on=["name", "country_id"],
    how="left",
).rename(columns={"id": "city_id"}).drop(columns=["name_y"])

places = places.rename(columns={"name_x": "name"})

places = places.merge(
    sections,
    left_on="section",
    right_on="name",
    how="left",
).rename(columns={"id": "section_id"}).drop(columns=["name_y"])

places = places.rename(columns={"name_x": "name"})

places = places.merge(
    categories,
    left_on=["category", "section_id"],
    right_on=["name", "section_id"],
    how="left",
).rename(columns={"id": "category_id"}).drop(columns=["name_y"])

places = places.rename(columns={"name_x": "name"})

places["id"] = range(1, len(places) + 1)

places = places[
    [
        "id",
        "source_id",
        "continent_id",
        "country_id",
        "city_id",
        "section_id",
        "category_id",
        "name",
        "address",
        "description",
        "url",
        "image_url",
    ]
]

In [17]:
import os

os.makedirs("csv", exist_ok=True)

continents.to_csv("csv/continents.csv", index=False)
countries.to_csv("csv/countries.csv", index=False)
cities.to_csv("csv/cities.csv", index=False)
sections.to_csv("csv/sections.csv", index=False)
categories.to_csv("csv/categories.csv", index=False)
places.to_csv("csv/places.csv", index=False)